# Task 4 — Kiểm tra dữ liệu FreshRetailNet-50K

Mục tiêu: kiểm tra dữ liệu trước khi tái lập bài báo. Chỉ đọc file đầu vào, không xóa hoặc sửa bản ghi.
Chạy lần lượt các ô. Task này dùng CPU, không cần GPU.

1. Đưa `train.csv` và `eval.csv` vào `MyDrive/PTDLKD/data/` (hoặc dùng hai file Parquet).
2. Chạy kết nối Drive rồi sửa ô cấu hình.
3. Chạy hàm kiểm tra, kiểm tra từng tập và đối chiếu hai tập.
4. Đọc kết quả, viết kết luận ở cuối, lưu notebook có output.

Nguồn: [dataset card](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K),
[repo](https://github.com/Dingdong-Inc/frn-50k-baseline).
Theo nguồn: train có 4.500.000 dòng; eval có 350.000 dòng; 19 cột.
Mốc ngày dưới đây đối chiếu cấu trúc 90 ngày lịch sử và 7 ngày đánh giá của repo.
Đó là **giá trị kỳ vọng**, không phải kết quả đã đo.

`ERROR`: vi phạm cấu trúc/miền hoặc tính nhất quán cần điều tra.
`REVIEW`: giá trị cần xác minh, chưa khẳng định sai.
`OK`: không thấy vi phạm trong phạm vi kiểm tra. `SKIP`: chưa đủ đầu vào để kiểm tra.
Các kiểm tra phụ thuộc chỉ tính trên số bản ghi đủ điều kiện (cột `evaluated`).


In [ ]:
from pathlib import Path
import sys, json, platform, importlib.util
from collections import defaultdict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display

try:
    ON_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    ON_COLAB = False
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
print('Python:', platform.python_version(), '| pandas:', pd.__version__, '| numpy:', np.__version__)


## 1. Cấu hình đường dẫn
Nếu dùng CSV: giữ `FILE_TYPE = 'csv'`. Nếu dùng Parquet: đổi thành `'parquet'` và đưa đúng hai file lên Drive.
Notebook chỉ kiểm tra định dạng đã chọn, không tự khẳng định CSV và Parquet giống nhau.
`BATCH_SIZE` giới hạn lượng dữ liệu đọc mỗi lần. Không dùng `nrows` để lấy mẫu cho kết quả bàn giao.


In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/PTDLKD/data')
OUTPUT_DIR = Path('/content/drive/MyDrive/PTDLKD/task4_results')
FILE_TYPE = 'csv'  # 'csv' hoặc 'parquet'
BATCH_SIZE = 20_000
ATOL = 1e-6  # Sai số tuyệt đối khi so tổng doanh số giờ với ngày (đơn vị đã chuẩn hóa)
RTOL = 1e-5  # Sai số tương đối; không dùng cho ID, cờ hoặc số giờ hết hàng

EXPECTED = {
    'train': {'rows': 4_500_000, 'days': 90, 'start': '2024-03-28', 'end': '2024-06-25'},
    'eval': {'rows': 350_000, 'days': 7, 'start': '2024-06-26', 'end': '2024-07-02'},
}
ID_COLS = ['city_id', 'store_id', 'management_group_id', 'first_category_id',
           'second_category_id', 'third_category_id', 'product_id']
LIST_COLS = ['hours_sale', 'hours_stock_status']
NUM_COLS = ID_COLS + ['sale_amount', 'stock_hour6_22_cnt', 'discount', 'holiday_flag',
                     'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level']
COLS = ID_COLS + ['dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt',
                 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag',
                 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level']
KEY = ['store_id', 'product_id', 'dt']
PATHS = {s: DATA_DIR / f'{s}.{FILE_TYPE}' for s in EXPECTED}
assert FILE_TYPE in ['csv', 'parquet']
assert OUTPUT_DIR.resolve() != DATA_DIR.resolve(), 'Hãy tách thư mục kết quả khỏi thư mục dữ liệu.'
for split, path in PATHS.items():
    assert path.is_file(), f'Không thấy {path}. Kiểm tra lại thư mục và tên file.'
    print(split, path, f'{path.stat().st_size / 1024**2:.1f} MiB')


## 2. Hàm kiểm tra
Đọc theo từng lô; chỉ giữ khóa của mọi bản ghi để kiểm tra trùng và chuỗi xuyên suốt các lô.
Không biến toàn bộ dữ liệu theo giờ thành một bảng lớn. CSV được đọc bằng bộ phân tích JSON
cho hai cột danh sách (không dùng `eval`). Sai định dạng sẽ được ghi nhận.

Doanh số bằng 0 được chấp nhận. Doanh số là giá trị đã chuẩn hóa, không diễn giải trực tiếp thành số món.
`hours_stock_status = 1` được hiểu là hết hàng theo mã nguồn. Chỉ số giờ hết hàng được cộng trên lát `[6:22]`.
Discount ngoài [0, 1] được gắn REVIEW; giới hạn này là quy tắc rà soát, không phải kết luận dữ liệu sai.


In [ ]:
checks, samples, profiles, sequences, coverage = [], [], [], {}, {}

def add_check(split, name, n, evaluated, severity='ERROR', note=''):
    n, evaluated = int(n), int(evaluated)
    status = 'SKIP' if evaluated == 0 else ('OK' if n == 0 else severity)
    checks.append(dict(split=split, check=name, violations=n, evaluated=evaluated,
                       status=status, note=note))

def batches(path):
    if path.suffix == '.csv':
        yield from pd.read_csv(path, chunksize=BATCH_SIZE)
    else:
        import pyarrow.parquet as pq
        for batch in pq.ParquetFile(path).iter_batches(batch_size=BATCH_SIZE):
            yield batch.to_pandas()

def array24(value):
    try:
        value = json.loads(value) if isinstance(value, str) else value
        if not isinstance(value, (list, tuple, np.ndarray)):
            return None
        arr = np.asarray(value, dtype=float)
        return arr if arr.shape == (24,) else None
    except (ValueError, TypeError, OverflowError):
        return None

def validate_split(split, path):
    spec = EXPECTED[split]
    totals, denominators = defaultdict(int), defaultdict(int)
    metadata, collected = {}, []
    sample_counts = defaultdict(int)
    row_count = 0

    def flag(name, mask, frame, eligible=None):
        mask = np.asarray(mask, dtype=bool)
        eligible = np.ones(len(frame), dtype=bool) if eligible is None else np.asarray(eligible, bool)
        bad = mask & eligible
        totals[name] += int(bad.sum())
        denominators[name] += int(eligible.sum())
        remain = max(0, 5 - sample_counts[name])
        idx = np.flatnonzero(bad)[:remain]
        for i in idx:
            row = frame.iloc[int(i)]
            sample = {'split': split, 'check': name, 'row_number': row_count + int(i) + 1}
            for col in KEY:
                sample[col] = str(row[col])
            sample['details'] = json.dumps({c: str(row[c])[:250] for c in frame.columns}, ensure_ascii=False)
            samples.append(sample)
        sample_counts[name] += len(idx)

    severity = {}
    def review(name):
        severity[name] = 'REVIEW'
        return name

    for batch_no, df in enumerate(batches(path), 1):
        missing_cols, extra_cols = set(COLS)-set(df.columns), set(df.columns)-set(COLS)
        if batch_no == 1:
            add_check(split, 'schema_columns', len(missing_cols)+len(extra_cols), 19,
                      note=f'Missing={sorted(missing_cols)}; extra={sorted(extra_cols)}')
        if missing_cols:
            raise ValueError(f'{split}: thiếu cột {sorted(missing_cols)}. Chưa thể hoàn tất kiểm tra.')
        flag('row_with_missing_or_blank', df[COLS].isna().any(axis=1) |
             df[COLS].map(lambda x: isinstance(x, str) and not x.strip()).any(axis=1), df)
        nums = {}
        for col in COLS:
            s = df[col]
            stat = metadata.setdefault(col, {'dtypes': set(), 'nulls': 0, 'min': None, 'max': None})
            stat['dtypes'].add(str(s.dtype))
            stat['nulls'] += int(s.isna().sum())
            if col in NUM_COLS:
                a = pd.to_numeric(s, errors='coerce').to_numpy(dtype=float)
                nums[col] = a
                finite = np.isfinite(a)
                flag('non_numeric_or_nonfinite:' + col, ~finite, df)
                if finite.any():
                    lo, hi = float(a[finite].min()), float(a[finite].max())
                    stat['min'] = lo if stat['min'] is None else min(stat['min'], lo)
                    stat['max'] = hi if stat['max'] is None else max(stat['max'], hi)
        key_valid = np.ones(len(df), dtype=bool)
        for col in ID_COLS:
            a = nums[col]
            bad = ~np.isfinite(a) | (a < 0) | (a != np.floor(a))
            flag('invalid_id:' + col, bad, df)
            if col in ['store_id', 'product_id']:
                key_valid &= ~bad

        raw_dates = df['dt'].astype('string')
        dates = pd.to_datetime(raw_dates, format='%Y-%m-%d', errors='coerce')
        date_valid = dates.notna().to_numpy() & raw_dates.str.fullmatch(r'\d{4}-\d{2}-\d{2}').fillna(False).to_numpy(bool)
        flag('invalid_date', ~date_valid, df)
        in_range = ((dates >= pd.Timestamp(spec['start'])) & (dates <= pd.Timestamp(spec['end']))).to_numpy()
        flag('date_outside_expected_split', ~in_range, df, date_valid)
        key_valid &= date_valid
        keys = pd.DataFrame({'store_id': nums['store_id'][key_valid].astype('int64'),
                             'product_id': nums['product_id'][key_valid].astype('int64'),
                             'dt': dates.to_numpy()[key_valid]})
        collected.append(keys)
        for col in ['holiday_flag', 'activity_flag']:
            flag('not_binary:' + col, ~np.isin(nums[col], [0, 1]), df, np.isfinite(nums[col]))
        a = nums['stock_hour6_22_cnt']
        flag('invalid_stock_count', (a < 0) | (a > 16) | (a != np.floor(a)), df, np.isfinite(a))
        flag('negative_daily_sales', nums['sale_amount'] < 0, df, np.isfinite(nums['sale_amount']))
        flag(review('discount_outside_0_1'), (nums['discount'] < 0) | (nums['discount'] > 1), df, np.isfinite(nums['discount']))
        flag(review('negative_precipitation'), nums['precpt'] < 0, df, np.isfinite(nums['precpt']))
        flag(review('humidity_outside_0_100'), (nums['avg_humidity'] < 0) | (nums['avg_humidity'] > 100), df, np.isfinite(nums['avg_humidity']))
        flag(review('negative_wind'), nums['avg_wind_level'] < 0, df, np.isfinite(nums['avg_wind_level']))

        arrays, valid_arrays = {}, {}
        for col in LIST_COLS:
            parsed = [array24(v) for v in df[col]]
            shape_ok = np.array([v is not None for v in parsed])
            flag('invalid_24_element_numeric_array:' + col, ~shape_ok, df)
            mat = np.full((len(df), 24), np.nan)
            if shape_ok.any():
                mat[shape_ok] = np.stack([v for v in parsed if v is not None])
            finite = np.isfinite(mat).all(axis=1)
            flag('nonfinite_array_values:' + col, ~finite, df, shape_ok)
            arrays[col], valid_arrays[col] = mat, shape_ok & finite
        sales, stocks = arrays['hours_sale'], arrays['hours_stock_status']
        flag('negative_hourly_sales', (sales < 0).any(axis=1), df, valid_arrays['hours_sale'])
        binary = np.isin(stocks, [0, 1]).all(axis=1)
        flag('not_binary:hours_stock_status', ~binary, df, valid_arrays['hours_stock_status'])
        flag('hourly_sum_vs_daily_sales', ~np.isclose(sales.sum(axis=1), nums['sale_amount'], atol=ATOL, rtol=RTOL),
             df, valid_arrays['hours_sale'] & np.isfinite(nums['sale_amount']))
        flag('stock_count_vs_hours_6_22', stocks[:, 6:22].sum(axis=1) != nums['stock_hour6_22_cnt'],
             df, valid_arrays['hours_stock_status'] & binary & np.isfinite(nums['stock_hour6_22_cnt']))
        row_count += len(df)
        if batch_no % 25 == 0:
            print(split, f'{row_count:,} dòng đã kiểm tra', flush=True)

    add_check(split, 'row_count_matches_reference', abs(row_count-spec['rows']), 1,
              note=f'Actual={row_count}; expected={spec["rows"]}; violations là chênh lệch số dòng.')
    if row_count == 0:
        raise ValueError(f'{split}: file không có dòng dữ liệu')
    for name in totals:
        add_check(split, name, totals[name], denominators[name], severity.get(name, 'ERROR'))
    for col, stat in metadata.items():
        profiles.append(dict(split=split, column=col, dtype='; '.join(sorted(stat['dtypes'])),
                             nulls=stat['nulls'], minimum=stat['min'], maximum=stat['max']))
    keys = pd.concat(collected, ignore_index=True)
    dup = keys.duplicated(KEY)
    add_check(split, 'duplicate_keys_excess_rows', dup.sum(), len(keys), note='Số dòng dư sau lần xuất hiện đầu tiên; chỉ xét khóa hợp lệ.')
    if dup.any():
        for row in keys.loc[dup].head(5).to_dict('records'):
            samples.append({'split': split, 'check': 'duplicate_keys_excess_rows', **row})
    g = keys.groupby(['store_id', 'product_id'], sort=False).agg(rows=('dt', 'size'), days=('dt', 'nunique'), start=('dt', 'min'), end=('dt', 'max'))
    g['complete'] = ((g['rows'] == spec['days']) & (g['days'] == spec['days']) &
                     (g['start'] == pd.Timestamp(spec['start'])) & (g['end'] == pd.Timestamp(spec['end'])))
    add_check(split, 'pair_count_matches_50000', abs(len(g)-50_000), 1, note=f'Actual={len(g)}; violations là chênh lệch số cặp.')
    add_check(split, 'incomplete_or_duplicated_series', (~g['complete']).sum(), len(g), note='Số cặp không đủ đúng các ngày kỳ vọng và đúng một dòng/ngày.')
    sequences[split] = g.reset_index()
    coverage[split] = keys.drop_duplicates(KEY)
    print(split, f'Hoàn tất {row_count:,} dòng, {len(g):,} cặp.', flush=True)


## 3. Kiểm tra toàn bộ hai tập
Chạy ô này trên CPU. Để ô chạy đến khi in “Hoàn tất” cho cả train và eval.
Mỗi lần chạy lại, kết quả trong bộ nhớ được khởi tạo lại để không cộng lặp.


In [ ]:
checks, samples, profiles, sequences, coverage = [], [], [], {}, {}
for split, path in PATHS.items():
    validate_split(split, path)
display(pd.DataFrame(checks))


## 4. Đối chiếu train với eval
Kiểm tra cặp cửa hàng–sản phẩm có khớp không và có trùng khóa ngày giữa hai tập không.
Hai tập có cùng cặp cửa hàng–sản phẩm là đúng kỳ vọng. Điều cần tránh là trùng cùng cặp **và cùng ngày**.


In [ ]:
# Loại kết quả cross_split cũ khi chạy lại riêng ô này.
checks[:] = [x for x in checks if x['split'] != 'cross_split']
train_pairs = pd.MultiIndex.from_frame(sequences['train'][['store_id', 'product_id']])
eval_pairs = pd.MultiIndex.from_frame(sequences['eval'][['store_id', 'product_id']])
only_train = train_pairs.difference(eval_pairs)
only_eval = eval_pairs.difference(train_pairs)
add_check('cross_split', 'pairs_only_in_train', len(only_train), len(train_pairs))
add_check('cross_split', 'pairs_only_in_eval', len(only_eval), len(eval_pairs))
overlap = coverage['train'].merge(coverage['eval'], on=KEY, how='inner')
add_check('cross_split', 'overlapping_keys', len(overlap), len(coverage['eval']))
summary = pd.DataFrame(checks)
display(summary)
display(pd.DataFrame(profiles))
display(pd.DataFrame(samples).head(20))


## 5. Xuất sản phẩm bàn giao
Mỗi lượt lưu vào thư mục mới theo thời điểm UTC, không ghi đè lượt trước.
`validation_summary.csv`: số vi phạm, số đủ điều kiện kiểm tra, trạng thái.
`column_profile.csv`: kiểu dữ liệu, null, min/max các cột số.
`issue_examples.csv`: tối đa 5 ví dụ cho mỗi loại lỗi mỗi tập (không phải toàn bộ bản ghi lỗi).
`series_train.csv` và `series_eval.csv`: độ đầy đủ của từng chuỗi.
`pairs_only_in_*.csv`, `overlapping_keys.csv`: đối chiếu hai tập.
`run_info.json`: đường dẫn file, dung lượng, cấu hình và phiên bản thư viện.

Giá trị minimum/maximum để trống ở cột không phải số là không áp dụng, không phải lỗi.
Nếu xuất hiện ERROR, xem ví dụ và xác minh nguyên nhân trước khi đề xuất sửa.
Nếu một kiểm tra SKIP hoặc dừng giữa chừng thì chưa thể kết luận kiểm tra đó đạt.


In [ ]:
RUN_DIR = OUTPUT_DIR / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
summary.to_csv(RUN_DIR / 'validation_summary.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(profiles).to_csv(RUN_DIR / 'column_profile.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(samples, columns=['split','check','row_number',*KEY,'details']).to_csv(RUN_DIR / 'issue_examples.csv', index=False, encoding='utf-8-sig')
for split, frame in sequences.items():
    frame.to_csv(RUN_DIR / f'series_{split}.csv', index=False, encoding='utf-8-sig')
only_train.to_frame(index=False).to_csv(RUN_DIR / 'pairs_only_in_train.csv', index=False)
only_eval.to_frame(index=False).to_csv(RUN_DIR / 'pairs_only_in_eval.csv', index=False)
overlap.to_csv(RUN_DIR / 'overlapping_keys.csv', index=False)
info = {'file_type': FILE_TYPE, 'batch_size': BATCH_SIZE, 'atol': ATOL, 'rtol': RTOL,
        'expected': EXPECTED, 'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__,
        'sources': {s: {'path': str(p), 'bytes': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns} for s,p in PATHS.items()},
        'sample_limit': '5 examples per rule per split', 'notes': 'Input files read only. No CSV/Parquet equivalence verification.'}
(RUN_DIR / 'run_info.json').write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu:', RUN_DIR)
print(summary['status'].value_counts().to_string())
display(summary.loc[summary['status'] != 'OK'])


## 6. Viết kết luận để bàn giao
Sau khi đọc output, thay các chỗ trống bằng số đã chạy:

“Nhóm kiểm tra toàn bộ … dòng train và … dòng eval, gồm … cột.
Khoảng ngày của train là … đến …; eval là … đến ….
Có … cặp cửa hàng–sản phẩm; … chuỗi không đủ cấu trúc ngày kỳ vọng.
Phát hiện … dòng dư do trùng khóa; … dòng thiếu/ô rỗng; … trường hợp không nhất quán dữ liệu theo giờ.
Các mục cần xác minh thêm gồm …; các kiểm tra chưa hoàn tất gồm ….
Dữ liệu [đáp ứng/chưa đáp ứng] các kiểm tra cấu trúc đã thực hiện. Đề xuất xử lý tiếp theo: ….”

Không kết luận “dữ liệu hoàn toàn chính xác” chỉ vì các kiểm tra trên đạt.
Bàn giao notebook có output, thư mục kết quả của lượt chạy đầy đủ và đoạn kết luận của nhóm.
